# Notebook 6 — Solutions to the exercises

Model solutions for instructors and self-learners. Try the exercises in `notebooks/06_real_data_m4raw.ipynb` first. The setup repeats Notebook 6 (download, preparation, λ choice); if you ran Notebook 6 in the same Colab session, the trained network is reused instead of retrained.

*Your numbers may differ slightly from any discussed below, depending on your hardware and library versions.*

## Setup (same as Notebook 6)

In [ ]:
%pip install -q huggingface_hub h5py   # usually already installed; ignore any message about restarting

import time
import urllib.request
import numpy as np
import matplotlib.pyplot as plt
import h5py
import torch
import torch.nn as nn
import torch.nn.functional as F
from huggingface_hub import hf_hub_download, list_repo_files
from skimage.metrics import structural_similarity

plt.rcParams["figure.dpi"] = 100
COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]  # line colors, always used in this order
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(0)
print(f"Using {device}.")

REPO, REVISION = "mylyu/M4Raw_brain", "v1.6.0"   # the M4Raw dataset on Hugging Face, fixed version

In [ ]:
# ---- From Notebooks 1-3 (now working on stacks of images too) ------------------
def fft2c(image):
    """Centered, orthonormal 2D FFT over the last two axes: image(s) -> k-space."""
    return np.fft.fftshift(np.fft.fft2(np.fft.ifftshift(image, axes=(-2, -1)), norm="ortho"), axes=(-2, -1))


def ifft2c(kspace):
    """Centered, orthonormal inverse 2D FFT over the last two axes: k-space -> image(s)."""
    return np.fft.fftshift(np.fft.ifft2(np.fft.ifftshift(kspace, axes=(-2, -1)), norm="ortho"), axes=(-2, -1))


def show(images, titles, cmap="gray", vmin=None, vmax=None):
    """Show images side by side."""
    n = len(images)
    fig, axes = plt.subplots(1, n, figsize=(3.4 * n, 3.6))
    for ax, im, title in zip(np.atleast_1d(axes), images, titles):
        ax.set_title(title)
        ax.imshow(im, cmap=cmap, vmin=vmin, vmax=vmax)
        ax.axis("off")
    plt.tight_layout()
    plt.show()


def nrmse(recon, reference):
    """Normalized root-mean-square error: 0 is perfect, 1 is as large as the image itself."""
    return np.linalg.norm(recon - reference) / np.linalg.norm(reference)


def tidy(ax):
    """Quiet grid and axes for line plots."""
    ax.grid(alpha=0.25, linewidth=0.6)
    ax.spines[["top", "right"]].set_visible(False)


def variable_density_lines(N, R, center_fraction=0.08, decay=3, seed=0):
    """N // R lines: a fully sampled center, the rest random and more likely near the center."""
    rng = np.random.default_rng(seed)
    n_total = N // R
    n_center = int(round(N * center_fraction))
    lines = np.zeros(N, dtype=bool)
    lines[N // 2 - n_center // 2 : N // 2 - n_center // 2 + n_center] = True
    candidates = np.flatnonzero(~lines)
    distance = np.abs(candidates - N // 2) / (N / 2)
    probability = (1 - distance) ** decay + 1e-6
    lines[rng.choice(candidates, size=n_total - n_center, replace=False,
                     p=probability / probability.sum())] = True
    return lines


# ---- Compressed sensing from Notebook 3, for comparison --------------------------
SQRT2 = np.sqrt(2)


def haar2(x, levels=4):
    c = x.astype(complex).copy()
    n = c.shape[0]
    for _ in range(levels):
        a = c[:n, :n]
        a = np.hstack([(a[:, 0::2] + a[:, 1::2]) / SQRT2, (a[:, 0::2] - a[:, 1::2]) / SQRT2])
        a = np.vstack([(a[0::2, :] + a[1::2, :]) / SQRT2, (a[0::2, :] - a[1::2, :]) / SQRT2])
        c[:n, :n] = a
        n //= 2
    return c


def ihaar2(c, levels=4):
    x = c.copy()
    n = x.shape[0] // 2 ** (levels - 1)
    for _ in range(levels):
        a = x[:n, :n]
        h = n // 2
        b = np.empty_like(a)
        b[0::2, :] = (a[:h, :] + a[h:, :]) / SQRT2
        b[1::2, :] = (a[:h, :] - a[h:, :]) / SQRT2
        a2 = np.empty_like(a)
        a2[:, 0::2] = (b[:, :h] + b[:, h:]) / SQRT2
        a2[:, 1::2] = (b[:, :h] - b[:, h:]) / SQRT2
        x[:n, :n] = a2
        n *= 2
    return x


def soft_threshold(c, lam):
    magnitude = np.abs(c)
    return c * np.maximum(magnitude - lam, 0) / np.maximum(magnitude, 1e-12)


def cs_reconstruct(kspace_measured, lines, lam=0.01, n_iter=200, levels=4):
    n = kspace_measured.shape[0]
    coarse = np.zeros((n, n), dtype=bool)
    coarse[: n // 2**levels, : n // 2**levels] = True
    x = ifft2c(kspace_measured)
    for _ in range(n_iter):
        k = np.where(lines[:, None], kspace_measured, fft2c(x))
        x = ifft2c(k)
        w = haar2(x, levels)
        x = ihaar2(np.where(coarse, w, soft_threshold(w, lam)), levels)
    return x

In [ ]:
def conv_block(c_in, c_out):
    """Two 3x3 convolutions, each followed by ReLU."""
    return nn.Sequential(
        nn.Conv2d(c_in, c_out, 3, padding=1), nn.ReLU(inplace=True),
        nn.Conv2d(c_out, c_out, 3, padding=1), nn.ReLU(inplace=True),
    )


class UNet(nn.Module):
    """A small U-Net that predicts a correction to the zero-filled image."""

    def __init__(self, channels=2, base=16):
        super().__init__()
        # encoder: features at 128, 64 and 32 pixels
        self.enc1 = conv_block(channels, base)
        self.enc2 = conv_block(base, 2 * base)
        self.enc3 = conv_block(2 * base, 4 * base)
        # bottleneck at 16 pixels
        self.bottom = conv_block(4 * base, 8 * base)
        # decoder: upsample, join with the matching encoder features (skip connection), convolve
        self.up3, self.dec3 = nn.ConvTranspose2d(8 * base, 4 * base, 2, stride=2), conv_block(8 * base, 4 * base)
        self.up2, self.dec2 = nn.ConvTranspose2d(4 * base, 2 * base, 2, stride=2), conv_block(4 * base, 2 * base)
        self.up1, self.dec1 = nn.ConvTranspose2d(2 * base, base, 2, stride=2), conv_block(2 * base, base)
        self.out = nn.Conv2d(base, channels, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(F.max_pool2d(e1, 2))
        e3 = self.enc3(F.max_pool2d(e2, 2))
        b = self.bottom(F.max_pool2d(e3, 2))
        d3 = self.dec3(torch.cat([self.up3(b), e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))
        return x + self.out(d1)          # residual: input + learned correction


def count_parameters(model):
    return sum(p.numel() for p in model.parameters())


def to_tensor(c):
    """Complex images (B, N, N) -> real tensor (B, 2, N, N)."""
    return torch.tensor(np.stack([c.real, c.imag], axis=1), dtype=torch.float32)


def to_complex(t):
    t = t.detach().cpu().numpy()
    return t[:, 0] + 1j * t[:, 1]


def tfft2c(x):
    return torch.fft.fftshift(torch.fft.fft2(torch.fft.ifftshift(x, dim=(-2, -1)), norm="ortho"), dim=(-2, -1))


def tifft2c(k):
    return torch.fft.fftshift(torch.fft.ifft2(torch.fft.ifftshift(k, dim=(-2, -1)), norm="ortho"), dim=(-2, -1))


def data_consistency(x, k_measured, mask):
    """As in Notebook 4, but the sampling mask is now an argument, so we can test other masks."""
    image = torch.complex(x[:, 0], x[:, 1])
    image = tifft2c(torch.where(mask, k_measured, tfft2c(image)))
    return torch.stack([image.real, image.imag], dim=1)


class SmallCNN(nn.Module):
    def __init__(self, channels=2, width=16, depth=5):
        super().__init__()
        layers = [nn.Conv2d(channels, width, 3, padding=1), nn.ReLU(inplace=True)]
        for _ in range(depth - 2):
            layers += [nn.Conv2d(width, width, 3, padding=1), nn.ReLU(inplace=True)]
        layers += [nn.Conv2d(width, channels, 3, padding=1)]
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return x + self.net(x)


class UnrolledNet(nn.Module):
    def __init__(self, cascades=4):
        super().__init__()
        self.cnns = nn.ModuleList([SmallCNN() for _ in range(cascades)])

    def forward(self, x, k_measured, mask):
        for cnn in self.cnns:
            x = data_consistency(cnn(x), k_measured, mask)
        return x


def load_pretrained(model, filename):
    """Download the weights trained in Notebook 4 from the module's GitHub repository."""
    url = f"https://raw.githubusercontent.com/anupbagale/mri-recon-tutorial/main/weights/{filename}"
    urllib.request.urlretrieve(url, filename)
    model.load_state_dict(torch.load(filename, map_location=device))
    return model.to(device).eval()


# The two networks trained on phantoms in Notebook 4
phantom_unet = load_pretrained(UNet(), "unet_r4.pt")
phantom_unrolled = load_pretrained(UnrolledNet(), "unrolled_r4.pt")
print("Loaded the phantom-trained U-Net and unrolled network from Notebook 4.")

In [ ]:
N_TRAIN_SUBJECTS, N_TEST_SUBJECTS = 20, 5

all_files = list_repo_files(REPO, repo_type="dataset", revision=REVISION)
# one T2-weighted scan (first repetition, "_T201") per subject; training and test subjects come from different splits
train_names = sorted(f for f in all_files if f.startswith("multicoil_train/") and f.endswith("_T201.h5"))[:N_TRAIN_SUBJECTS]
test_names = sorted(f for f in all_files if f.startswith("multicoil_val/") and f.endswith("_T201.h5"))[:N_TEST_SUBJECTS]


def download(name):
    """Download one scan (about 12 MB) from Hugging Face, or reuse it if it is already here."""
    return hf_hub_download(REPO, filename=name, repo_type="dataset", revision=REVISION)


start = time.time()
train_paths = [download(name) for name in train_names]
test_paths = [download(name) for name in test_names]
print(f"{len(train_paths)} training and {len(test_paths)} test scans ready ({time.time() - start:.0f} s)")
print("first test scan:", test_names[0])

In [ ]:
with h5py.File(test_paths[0], "r") as hf:
    print("keys:      ", list(hf.keys()))
    print("attributes:", dict(hf.attrs))
    volume_kspace = hf["kspace"][()]
    stored_rss = hf["reconstruction_rss"][()]
acquired = (np.abs(volume_kspace) ** 2).sum(axis=(0, 1, 2)) > 0

In [ ]:
def load_volume(path):
    """Multi-coil k-space of one scan, with the two image axes swapped so that phase encoding
    runs vertically (along rows), as in Notebooks 2-5. The head therefore appears rotated."""
    with h5py.File(path, "r") as hf:
        return hf["kspace"][()].swapaxes(-1, -2)


def coil_combine(coil_kspace, n_center=24):
    """Combine coil images into one complex image, weighting each coil by its estimated sensitivity.
    The sensitivities are estimated from a low-resolution image made from the central k-space lines."""
    coil_images = ifft2c(coil_kspace)
    n = coil_kspace.shape[-2]
    center = slice(n // 2 - n_center // 2, n // 2 + n_center // 2)
    low = np.zeros_like(coil_kspace)
    low[:, center, :] = coil_kspace[:, center, :]
    low_images = ifft2c(low)
    sensitivities = low_images / (np.sqrt((np.abs(low_images) ** 2).sum(axis=0, keepdims=True)) + 1e-12)
    return (np.conj(sensitivities) * coil_images).sum(axis=0)


def prepare(paths, skip=3):
    """Coil-combined complex images of the middle slices of each scan, scaled so that the brightest
    0.5% of pixels are around 1. The first and last few slices contain little brain and are skipped."""
    images = []
    for path in paths:
        kspace = load_volume(path)
        for s in range(skip, kspace.shape[0] - skip):
            image = coil_combine(kspace[s])
            images.append(image / np.percentile(np.abs(image), 99.5))
    return np.stack(images).astype(np.complex64)


start = time.time()
train_images = prepare(train_paths)
test_images = prepare(test_paths)
print(f"{len(train_images)} training slices, {len(test_images)} test slices ({time.time() - start:.0f} s)")

In [ ]:
def sampling_lines(acquired, R, center_fraction=0.08, decay=3, seed=0):
    """Variable-density sampling as in Notebook 2, but choosing only among the lines the scanner
    actually acquired. R is counted relative to the acquired lines."""
    rng = np.random.default_rng(seed)
    n = acquired.size
    available = np.flatnonzero(acquired)
    n_total = len(available) // R
    n_center = int(round(len(available) * center_fraction))
    lines = np.zeros(n, dtype=bool)
    lines[n // 2 - n_center // 2 : n // 2 - n_center // 2 + n_center] = True
    candidates = available[~lines[available]]
    distance = np.abs(candidates - n // 2) / (len(available) / 2)
    probability = np.clip(1 - distance, 0, None) ** decay + 1e-6
    lines[rng.choice(candidates, size=n_total - n_center, replace=False, p=probability / probability.sum())] = True
    return lines


R = 4
lines = sampling_lines(acquired, R)
print(f"Measuring {lines.sum()} of {acquired.sum()} acquired lines (R = {R})")

train_measured = fft2c(train_images) * lines[:, None]
test_measured = fft2c(test_images) * lines[:, None]
train_zf, test_zf = ifft2c(train_measured), ifft2c(test_measured)
train_truth, test_truth = np.abs(train_images), np.abs(test_images)

In [ ]:
def ssim(recon, reference):
    return structural_similarity(reference, recon, data_range=1.0)


def scores(recons, truths):
    """Mean NRMSE and SSIM over a set of slices."""
    return (np.mean([nrmse(r, t) for r, t in zip(recons, truths)]),
            np.mean([ssim(r, t) for r, t in zip(recons, truths)]))


# Choose λ on a few TRAINING slices, never on the test slices
tune = range(0, len(train_images), max(1, len(train_images) // 4))
lam_values = [0.001, 0.002, 0.005, 0.01, 0.02, 0.05]
lam_errors = []
for lam in lam_values:
    errors = [nrmse(np.abs(cs_reconstruct(train_measured[j], lines, lam)), train_truth[j]) for j in tune]
    lam_errors.append(np.mean(errors))
    print(f"λ = {lam:<6} NRMSE on {len(errors)} training slices: {lam_errors[-1]:.3f}")
best_lam = lam_values[int(np.argmin(lam_errors))]
print(f"Chosen λ = {best_lam}")

In [ ]:
def to_tensors(zf, measured, truth):
    x = torch.tensor(np.stack([zf.real, zf.imag], axis=1), dtype=torch.float32)
    k = torch.tensor(measured, dtype=torch.complex64)
    y = torch.tensor(np.stack([truth, np.zeros_like(truth)], axis=1), dtype=torch.float32)
    return x, k, y


def train_unrolled(model, epochs, batch_size=8, lr=1e-3, seed=0, n_val=24):
    """Train an unrolled network on the real training slices; the last n_val slices
    (from the last two training subjects) are held out for validation."""
    torch.manual_seed(seed)
    model.to(device)
    mask = torch.tensor(lines[:, None], device=device)
    X, K, Y = to_tensors(train_zf, train_measured, train_truth)
    n_fit = len(X) - n_val
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    history = {"train": [], "val": []}
    for epoch in range(epochs):
        start = time.time()
        model.train()
        order = np.random.default_rng(seed + epoch).permutation(n_fit)
        total = 0.0
        for b in range(0, n_fit, batch_size):
            idx = order[b : b + batch_size]
            out = model(X[idx].to(device), K[idx].to(device), mask)
            # compare magnitudes: the reference is complex, and its leftover phase carries no anatomy
            loss = F.l1_loss(torch.sqrt(out[:, 0] ** 2 + out[:, 1] ** 2 + 1e-12), Y[idx, 0].to(device))
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total += loss.item() * len(idx)
        model.eval()
        with torch.no_grad():
            out = model(X[n_fit:].to(device), K[n_fit:].to(device), mask)
            val_loss = F.l1_loss(torch.sqrt(out[:, 0] ** 2 + out[:, 1] ** 2 + 1e-12), Y[n_fit:, 0].to(device)).item()
        history["train"].append(total / n_fit)
        history["val"].append(val_loss)
        print(f"epoch {epoch + 1}/{epochs}: train loss {total / n_fit:.4f}, validation loss {val_loss:.4f}  ({time.time() - start:.0f} s)")
    return history


def predict(model, zf, measured, mask_lines=None):
    """Magnitude reconstructions from a trained network (U-Net or unrolled)."""
    mask_lines = lines if mask_lines is None else mask_lines
    model.eval()
    x = torch.tensor(np.stack([zf.real, zf.imag], axis=1), dtype=torch.float32).to(device)
    k = torch.tensor(measured, dtype=torch.complex64).to(device)
    mask = torch.tensor(mask_lines[:, None], device=device)
    outputs = []
    with torch.no_grad():
        for b in range(0, len(x), 8):
            out = model(x[b:b + 8], k[b:b + 8], mask) if isinstance(model, UnrolledNet) else model(x[b:b + 8])
            outputs.append(torch.sqrt(out[:, 0] ** 2 + out[:, 1] ** 2).cpu().numpy())
    return np.concatenate(outputs)

In [ ]:
import os
real_unrolled = UnrolledNet().to(device)
if os.path.exists("unrolled_m4raw_r4.pt"):
    real_unrolled.load_state_dict(torch.load("unrolled_m4raw_r4.pt", map_location=device))
    print("Loaded the real-data network saved by Notebook 6.")
else:
    print("No saved network found; training it as in Notebook 6.")
    train_unrolled(real_unrolled, epochs=10)
    torch.save(real_unrolled.state_dict(), "unrolled_m4raw_r4.pt")

## Exercise 1: higher acceleration on real data

In [ ]:
test_subset = list(range(0, len(test_images), max(1, len(test_images) // 10)))[:10]
print(f"{'R':>3}{'zero-filled':>14}{'compressed sensing':>21}{'unrolled (real)':>18}   (NRMSE)")
for R_new in [4, 6, 8]:
    new_lines = sampling_lines(acquired, R_new)
    measured_new = fft2c(test_images[test_subset]) * new_lines[:, None]
    zf_new = ifft2c(measured_new)
    truth_new = test_truth[test_subset]
    zf_err = np.mean([nrmse(np.abs(z), t) for z, t in zip(zf_new, truth_new)])
    cs_err = np.mean([nrmse(np.abs(cs_reconstruct(m, new_lines, best_lam)), t) for m, t in zip(measured_new, truth_new)])
    net_err = np.mean([nrmse(r, t) for r, t in zip(predict(real_unrolled, zf_new, measured_new, new_lines), truth_new)])
    print(f"{R_new:>3}{zf_err:>14.3f}{cs_err:>21.3f}{net_err:>18.3f}")

**What to look for:** every method degrades as R grows, because each scan simply contains less information. The interesting question is how the **gaps** change. The network was trained only at R = 4, with one mask; its data consistency step uses the new mask, but its learned steps have never seen these artifact patterns. If its advantage over compressed sensing shrinks at R = 8, that is the distribution shift of Notebook 5 again. In practice, networks are trained on several accelerations and random masks, so that one network handles them all.

## Exercise 2: transfer learning

In [ ]:
import copy

fine_tuned = copy.deepcopy(phantom_unrolled)
print("Starting from the phantom-trained network:")
fine_tuned_history = train_unrolled(fine_tuned, epochs=3)
from_scratch = UnrolledNet()
print("Starting from random weights:")
scratch_history = train_unrolled(from_scratch, epochs=3)

subset = list(range(0, len(test_images), 3))
for name, model in [("fine-tuned from phantoms, 3 epochs", fine_tuned),
                    ("from scratch, 3 epochs", from_scratch),
                    ("from scratch, 10 epochs (Notebook 6)", real_unrolled)]:
    e, s_ = scores(predict(model, test_zf[subset], test_measured[subset]), test_truth[subset])
    print(f"{name:<38} NRMSE {e:.3f}   SSIM {s_:.3f}")

**What to look for:** compare the two 3-epoch networks. Starting from phantoms gives the network a head start only if what it learned (removing incoherent aliasing while keeping the measured data) also applies to real anatomy. Whatever the outcome, note that both 3-epoch networks are compared with the same budget; comparing a 3-epoch network with the 10-epoch one would not be fair (Notebook 4, Exercise 1). **Transfer learning**, pre-training on plentiful data and then fine-tuning on scarce data, is widely used in medical imaging, where real training data is expensive.

## Exercise 3: a cleaner reference

In [ ]:
first = test_names[0]
repetition_names = [first] + [first.replace("_T201", tag) for tag in ["_T202", "_T203"]]
repetition_names = [n for n in repetition_names if n in all_files]
print("repetitions available:", repetition_names)

volumes = [load_volume(download(n)) for n in repetition_names]
n_slices = volumes[0].shape[0]
slices = range(3, n_slices - 3)
single, averaged = [], []
for s in slices:
    reps = [coil_combine(v[s]) for v in volumes]
    scale = np.percentile(np.abs(reps[0]), 99.5)          # the same scale for every repetition
    single.append(np.abs(reps[0]) / scale)
    averaged.append(np.abs(np.mean(reps, axis=0)) / scale)
single, averaged = np.stack(single), np.stack(averaged)

mid = len(single) // 2
show([single[mid], averaged[mid]], ["one repetition", f"average of {len(volumes)}"], vmin=0, vmax=1)

# background noise: the standard deviation in the image corners
corner = (slice(0, 20), slice(0, 20))
print(f"noise in the corners: one repetition {single[mid][corner].std():.4f}, average {averaged[mid][corner].std():.4f}")

# the first test subject's slices are the first len(slices) test slices
k = len(slices)
subject_zf, subject_measured = test_zf[:k], test_measured[:k]
methods = {
    "zero-filled": np.abs(subject_zf),
    "compressed sensing": np.stack([np.abs(cs_reconstruct(m, lines, best_lam)) for m in subject_measured]),
    "unrolled (phantoms)": predict(phantom_unrolled, subject_zf, subject_measured),
    "unrolled (real data)": predict(real_unrolled, subject_zf, subject_measured),
}
print(f"{'method':<24}{'NRMSE vs single':>17}{'vs average':>12}{'SSIM vs single':>16}{'vs average':>12}")
for name, recon in methods.items():
    e1, s1 = scores(recon, single)
    e2, s2 = scores(recon, averaged)
    print(f"{name:<24}{e1:>17.3f}{e2:>12.3f}{s1:>16.3f}{s2:>12.3f}")

Averaging three repetitions with independent noise reduces the noise by a factor of about √3 ≈ 1.7 (the M4Raw authors note the same theoretical gain), which you can check in the corner measurement. The anatomy stays the same, as long as the subject did not move between repetitions.

**What to look for in the table:** methods that remove noise, such as compressed sensing and the networks, were penalized by the noisy single-repetition reference, so their scores should improve the most against the averaged one. Zero-filling keeps the noise of the scan, so its score should change least. If the ranking of the methods changes, the noisy reference was hiding real differences. This is why careful studies use a better reference, or radiologists, rather than trusting one noisy "ground truth".

## Exercise 4: a different contrast

In [ ]:
flair_names = [n.replace("_T201", "_FLAIR01") for n in test_names]
flair_names = [n for n in flair_names if n in all_files]
print(f"{len(flair_names)} FLAIR test scans available")
flair_images = prepare([download(n) for n in flair_names])
flair_measured = fft2c(flair_images) * lines[:, None]
flair_zf, flair_truth = ifft2c(flair_measured), np.abs(flair_images)

subset = list(range(0, len(flair_images), 3))
flair_results = {
    "zero-filled": np.abs(flair_zf[subset]),
    "compressed sensing": np.stack([np.abs(cs_reconstruct(flair_measured[j], lines, best_lam)) for j in subset]),
    "unrolled (trained on T2)": predict(real_unrolled, flair_zf[subset], flair_measured[subset]),
}
for name, recon in flair_results.items():
    e, s_ = scores(recon, flair_truth[subset])
    print(f"{name:<26} NRMSE {e:.3f}   SSIM {s_:.3f}")
j = len(subset) // 2
show([flair_truth[subset][j]] + [flair_results[n][j] for n in flair_results], ["FLAIR reference"] + list(flair_results), vmin=0, vmax=1)

**What to look for:** compressed sensing makes no assumption about contrast, so it should work about as well on FLAIR as on T2. The network learned T2 images: bright fluid, particular tissue contrasts. On FLAIR, where fluid is dark, it faces a distribution shift. Data consistency limits how far it can go wrong, but compare the error and the images carefully. Real reconstruction networks are usually trained on all the contrasts they will be used for.